In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: Eine Qiskit Function von Qedma
*Siehe die [API-Referenz](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Qiskit Functions sind ein experimentelles Feature, das ausschließlich Nutzern des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über die IBM Quantum Platform API) Plan zur Verfügung steht. Sie befinden sich im Preview-Release-Status und können sich noch ändern.

## Überblick
Obwohl Quantenprozessoren (QPUs) in den letzten Jahren erhebliche Fortschritte gemacht haben, bleiben Fehler durch Rauschen und Unvollkommenheiten in bestehender Hardware eine zentrale Herausforderung für Entwickler von Quantenalgorithmen. Da sich das Feld Quantenberechnungen im Utility-Maßstab nähert, die klassisch nicht mehr verifiziert werden können, gewinnen Lösungen zur fehlerfreien Berechnung mit garantierter Genauigkeit zunehmend an Bedeutung. Um dieser Herausforderung zu begegnen, hat Qedma Quantum Error Mitigation (QESEM) entwickelt, das nahtlos in die IBM Quantum Platform als [Qiskit Function](/guides/functions) integriert ist.

Mit QESEM kannst du Quantenschaltkreise auf verrauschten QPUs ausführen und hochgenaue, fehlerfreie Ergebnisse mit sehr effizienten QPU-Zeit-Overheads erzielen, die nahe an fundamentalen Grenzen liegen. Dazu nutzt QESEM eine Reihe proprietärer Methoden von Qedma zur Charakterisierung und Reduktion von Fehlern. Fehlerreduktionstechniken umfassen Gate-Optimierung, rauschbewusste Transpilation, Error Suppression (ES) und unbiasiertes Error Mitigation (EM). Durch diese Kombination charakterisierungsbasierter Methoden kannst du zuverlässige, fehlerfreie Ergebnisse für generische großvolumige Quantenschaltkreise erzielen und so Anwendungen erschließen, die sonst nicht realisierbar wären.

Eine vollständige Beschreibung der zugrunde liegenden Komponenten sowie eine Demonstration im Utility-Maßstab findest du im Artikel [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Beschreibung
Du kannst die QESEM-Funktion von Qedma verwenden, um deine Schaltkreise mit Error Suppression und Mitigation einfach zu schätzen und auszuführen, größere Schaltkreisvolumen und höhere Genauigkeiten zu erreichen. Zur Nutzung von QESEM gibst du einen Quantenschaltkreis, eine Menge zu messender Observablen, eine Zielstatistikgenauigkeit für jede Observable und einen gewählten QPU an. Bevor du den Schaltkreis auf die Zielgenauigkeit ausführst, kannst du die erforderliche QPU-Zeit auf Basis einer analytischen Berechnung schätzen, die keine Schaltkreisausführung erfordert. Sobald du mit der QPU-Zeitschätzung zufrieden bist, kannst du den Schaltkreis mit QESEM ausführen.

Bei der Ausführung eines Schaltkreises führt QESEM ein auf deinen Schaltkreis zugeschnittenes Gerätcharakterisierungsprotokoll durch, das ein zuverlässiges Rauschmodell für die im Schaltkreis auftretenden Fehler liefert. Auf Basis der Charakterisierung implementiert QESEM zunächst eine rauschbewusste Transpilation, um den Eingangsschaltkreis auf eine Menge physikalischer Qubits und Gates abzubilden, die das Rauschen bezüglich der Zielobservable minimiert. Dazu gehören die nativ verfügbaren Gates (CX/CZ auf IBM&reg;-Geräten) sowie zusätzliche, von QESEM optimierte Gates, die den erweiterten Gate-Satz von QESEM bilden. QESEM führt dann eine Reihe charakterisierungsbasierter ES- und EM-Schaltkreise auf dem QPU aus und erfasst deren Messergebnisse. Diese werden anschließend klassisch nachverarbeitet, um einen unverzerrten Erwartungswert und einen Fehlerbalken für jede Observable entsprechend der angeforderten Genauigkeit bereitzustellen.

![Qedma QESEM Überblick](../docs/images/guides/qedma-qesem/overview.svg)
QESEM hat nachweislich hochgenaue Ergebnisse für eine Vielzahl von Quantenanwendungen und bei den größten heute erreichbaren Schaltkreisvolumen geliefert. QESEM bietet die folgenden nutzerorientierten Features, die im nachfolgenden Benchmarks-Abschnitt demonstriert werden:
-	**Garantierte Genauigkeit:** QESEM liefert unverzerrte Schätzungen für Erwartungswerte von Observablen. Die EM-Methode ist mit theoretischen Garantien ausgestattet, die – zusammen mit Qedmas modernster Charakterisierung – sicherstellen, dass die Mitigation bis zur nutzerdefinierten Genauigkeit gegen die rauschfreie Schaltkreisausgabe konvergiert. Im Gegensatz zu vielen heuristischen EM-Methoden, die anfällig für systematische Fehler oder Verzerrungen sind, ist QESEMs garantierte Genauigkeit entscheidend für zuverlässige Ergebnisse bei generischen Quantenschaltkreisen und Observablen.
-	**Skalierbarkeit auf große QPUs:** Die QPU-Zeit von QESEM hängt vom Schaltkreisvolumen ab, ist aber ansonsten unabhängig von der Anzahl der Qubits. Qedma hat QESEM auf den größten heute verfügbaren Quantengeräten demonstriert, darunter die IBM Quantum 127-Qubit Eagle- und 133-Qubit Heron-Geräte.
-	**Anwendungsunabhängig:** QESEM wurde für eine Vielzahl von Anwendungen demonstriert, darunter Hamiltonian Simulation, VQE, QAOA und Amplitudenabschätzung. Nutzer können beliebige Quantenschaltkreise und Observablen eingeben und genaue, fehlerfreie Ergebnisse erhalten. Die einzigen Einschränkungen werden durch die Hardware-Spezifikationen und die zugewiesene QPU-Zeit bestimmt, die die zugänglichen Schaltkreisvolumen und Ausgabegenauigkeiten festlegen. Im Gegensatz dazu sind viele Fehlerreduktionstechniken anwendungsspezifisch oder beinhalten unkontrollierte Heuristiken, die sie für generische Quantenschaltkreise und Anwendungen ungeeignet machen.
-  **Erweiterter Gate-Satz:** QESEM unterstützt Gates mit gebrochenen Winkeln und stellt Qedma-optimierte $Rzz(\theta)$-Gates mit gebrochenen Winkeln auf IBM Quantum Heron- und Eagle-Geräten bereit. Dieser erweiterte Gate-Satz ermöglicht eine effizientere Kompilierung und erschließt Schaltkreisvolumen, die um einen Faktor von bis zu 2 größer sind als bei der Standard-CX/CZ-Kompilierung.
-	**Multibasige Observablen:** QESEM unterstützt Eingabe-Observablen, die aus vielen nicht-kommutierenden Pauli-Strings bestehen, wie generische Hamiltonians. Die Wahl der Messbasen und die Optimierung der QPU-Ressourcenallokation (Shots und Schaltkreise) wird dann automatisch von QESEM durchgeführt, um die erforderliche QPU-Zeit für die angeforderte Genauigkeit zu minimieren. Diese Optimierung, die Hardware-Fidelities und Ausführungsraten berücksichtigt, ermöglicht es dir, tiefere Schaltkreise auszuführen und höhere Genauigkeiten zu erzielen.
## Benchmarks
QESEM wurde für eine große Bandbreite von Anwendungsfällen getestet. Die folgenden Beispiele können dir helfen, einzuschätzen, welche Arten von Workloads du mit QESEM ausführen kannst.

Eine wichtige Kennzahl zur Quantifizierung der Schwierigkeit von sowohl Error Mitigation als auch klassischer Simulation für einen gegebenen Schaltkreis und eine Observable ist das **aktive Volumen**: die Anzahl der CNOT-Gates, die die Observable im Schaltkreis beeinflussen. Das aktive Volumen hängt von der Schaltkreistiefe und -breite, dem Gewicht der Observable und der Schaltkreisstruktur ab, die den Lichtkegel der Observable bestimmt. Weitere Details findest du im Vortrag vom [2024 IBM Quantum Summit](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM bietet besonders großen Mehrwert im Hochvolumenbereich und liefert zuverlässige Ergebnisse für generische Schaltkreise und Observablen.

![Aktives Volumen](../docs/images/guides/qedma-qesem/active_volume.svg)

| Anwendung    | Anzahl Qubits | Gerät | Schaltkreisbeschreibung | Genauigkeit | Gesamtzeit | Runtime-Nutzung |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE-Schaltkreis  | 8              | Eagle (r3) | 21 Gesamtschichten, 9 Messbasen, 1D-Kette                    | 98%      | 35 Min.      | 14 Min.         |
| Kicked Ising   | 28              | Eagle (r3) | 3 einzigartige Schichten x 3 Schritte, 2D Heavy-Hex-Topologie                      | 97%     | 22 Min.      | 4 Min.          |
| Kicked Ising   | 28              | Eagle (r3) | 3 einzigartige Schichten x 8 Schritte, 2D Heavy-Hex-Topologie                     | 97%      | 116 Min.      | 23 Min.          |
| Trotterisierte Hamiltonian-Simulation   | 40  | Eagle (r3)            | 2 einzigartige Schichten x 10 Trotter-Schritte, 1D-Kette                    | 97%      | 3 Std.     | 25 Min.         |
| Trotterisierte Hamiltonian-Simulation   | 119  | Eagle (r3)           | 3 einzigartige Schichten x 9 Trotter-Schritte, 2D Heavy-Hex-Topologie                    | 95%      | 6,5 Std.     | 45 Min.         |
| Kicked Ising  | 136             | Heron (r2) | 3 einzigartige Schichten x 15 Schritte, 2D Heavy-Hex-Topologie                    | 99%      | 52 Min.             | 9 Min.           |

Die Genauigkeit wird hier relativ zum Idealwert der Observable gemessen: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, wobei „$\epsilon$" die absolute Präzision der Mitigation (vom Nutzer festgelegt) ist und $\langle O \rangle_{ideal}$ den Erwartungswert des rauschfreien Schaltkreises bezeichnet.
„Runtime-Nutzung" misst die Nutzung des Benchmarks im Batch-Modus (Summe der Nutzung einzelner Jobs), während „Gesamtzeit" die Nutzung im Session-Modus (Experiment-Wandzeit) misst, die zusätzliche klassische und Kommunikationszeiten einschließt. QESEM ist für die Ausführung in beiden Modi verfügbar, sodass Nutzer ihre verfügbaren Ressourcen optimal nutzen können.

Die 28-Qubit Kicked-Ising-Schaltkreise simulieren den von Shinjo et al. untersuchten Discrete Time Quasicrystal (siehe [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) und [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) auf drei verbundenen Schleifen von ibm_kawasaki. Die hier verwendeten Schaltkreisparameter sind $(\theta_x, \theta_z) = (0.9 \pi, 0)$ mit einem ferromagnetischen Anfangszustand $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Die gemessene Observable ist der Absolutbetrag der Magnetisierung $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Das Kicked-Ising-Experiment im Utility-Maßstab wurde auf den 136 besten Qubits von ibm_fez ausgeführt; dieser spezifische Benchmark wurde beim Clifford-Winkel $(\theta_x, \theta_z) = (\pi, 0)$ ausgeführt, bei dem das aktive Volumen mit der Schaltkreistiefe langsam wächst – was zusammen mit den hohen Geräte-Fidelities eine hohe Genauigkeit bei kurzer Laufzeit ermöglicht.

Trotterisierte Hamiltonian-Simulations-Schaltkreise gelten für ein Transverse-Field-Ising-Modell bei gebrochenen Winkeln: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ und $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ entsprechend (siehe [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Der Schaltkreis im Utility-Maßstab wurde auf den 119 besten Qubits von ibm_brisbane ausgeführt, während das 40-Qubit-Experiment auf der besten verfügbaren Kette ausgeführt wurde. Die Genauigkeit wird für die Magnetisierung angegeben; hochgenaue Ergebnisse wurden auch für höhergewichtige Observablen erzielt.

Der VQE-Schaltkreis wurde zusammen mit Forschern des Center for Quantum Technology and Applications am Deutschen Elektronen-Synchrotron (DESY) entwickelt. Die Zielobservable war hier ein Hamiltonian, der aus einer großen Anzahl nicht-kommutierender Pauli-Strings besteht, was QESEMs optimierte Leistung für multibasige Observablen unterstreicht. Die Mitigation wurde auf einen klassisch optimierten Ansatz angewendet; obwohl diese Ergebnisse noch unveröffentlicht sind, werden Ergebnisse gleicher Qualität für verschiedene Schaltkreise mit ähnlichen strukturellen Eigenschaften erzielt.
## Erste Schritte
Authentifiziere dich mit deinem [IBM Quantum Platform API-Schlüssel](http://quantum.cloud.ibm.com/) und wähle die QESEM Qiskit Function wie folgt aus. (Dieses Snippet setzt voraus, dass du dein Konto bereits [in deiner lokalen Umgebung gespeichert hast](/guides/functions#install-qiskit-functions-catalog-client).)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Du kannst die vertrauten Qiskit Serverless APIs verwenden, um den Status deines Qiskit Function Workloads zu prüfen oder Ergebnisse abzurufen:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Das folgende Code-Snippet zeigt, wie die QPU-Zeitschätzung abgerufen wird (wenn `estimate_time_only` gesetzt ist):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


Das folgende Code-Snippet demonstriert, wie die Mitigationsergebnisse (wenn `estimate_time_only` nicht gesetzt ist) und Ausführungsmetriken abgerufen werden. Diese enthalten wesentliche Daten, die ein tieferes Verständnis davon ermöglichen, wie verschiedene Parameter die QESEM-Ausführung beeinflussen. Sie können auch relevant sein, wenn du einen Artikel auf Basis deiner Forschung verfasst.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## Fehlermeldungen abrufen
Wenn der Status deines Workloads ERROR ist, verwende `job.result()`, um die Fehlermeldung wie folgt abzurufen:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Support erhalten

Das Qedma-Support-Team ist für dich da! Wenn du auf Probleme stößt oder Fragen zur Verwendung der QESEM Qiskit Function hast, zögere nicht, uns zu kontaktieren. Unser kompetentes und freundliches Support-Team steht dir bei technischen Anliegen oder Fragen jeder Art zur Verfügung.

Du kannst uns für Unterstützung unter support@qedma.com per E-Mail erreichen. Bitte gib so viele Details wie möglich zum aufgetretenen Problem an, damit wir dir schnell und präzise helfen können. Du kannst auch deinen dedizierten Qedma-POC-Ansprechpartner per E-Mail oder Telefon kontaktieren.

Um uns zu ermöglichen, dir effizienter zu helfen, stelle uns bitte folgende Informationen bereit, wenn du uns kontaktierst:

- Eine detaillierte Beschreibung des Problems
- Die Job-ID
- Alle relevanten Fehlermeldungen oder -codes

Wir sind bestrebt, dir prompten und effektiven Support zu bieten, damit du mit unserer Qiskit Function die bestmögliche Erfahrung machst.

Wir sind stets bestrebt, unser Produkt zu verbessern, und freuen uns über deine Vorschläge! Wenn du Ideen hast, wie wir unsere Dienste oder Features verbessern können, sende uns deine Gedanken an support@qedma.com.

## Nächste Schritte

> **Tip:** - [Zugang zu Qedma QESEM beantragen](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - Besuche die [API-Referenz](https://docs.quantum.ibm.com/api/functions/qedma-qesem) für diese Qiskit Function.
> - Lies [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - Lies [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).